# Data Drift em Stream: Eficácia de Estratégias de Retreinamento em Função do Tipo, Severidade e Classificador

**TCC – Engenharia de Software, FGA/UnB** · Gabriela Silva Alves · Orientador: Isaque Alves · Coorientadora: Carla Rocha

Este notebook implementa o experimento fatorial descrito no Capítulo 2 da monografia:

| Fator | Níveis |
|---|---|
| Tipo de drift | abrupto, gradual, incremental, recorrente |
| Severidade ($H^2$ alvo) | baixa (0,1), média (0,4), alta (0,7) |
| Classificador | Naive Bayes, Random Forest, XGBoost |
| Estratégia | reset completo, janela deslizante, incremental, reativação de modelos históricos, sem retreinamento (controle) |

$4 \times 3 \times 3 \times 5 = 180$ condições, cada uma com `N_REPS` repetições (sementes distintas).

**Como executar:** `Ambiente de execução → Executar tudo`. Com `N_REPS = 10` a execução completa leva poucos minutos no Colab. Todos os resultados (CSVs, figuras em PDF/PNG e tabelas LaTeX) são gravados em `resultados/` e compactados em `resultados.zip` ao final.

## 0. Configuração do ambiente

In [ ]:
import sys, subprocess, importlib.util
for pkg in ["xgboost", "statsmodels"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, json, platform
import matplotlib.pyplot as plt
import matplotlib as mpl
from joblib import Parallel, delayed

N_REPS  = int(os.environ.get("N_REPS", 10))   # repetições por condição
N_JOBS  = -1                                  # núcleos para paralelizar
OUT     = "resultados"
os.makedirs(f"{OUT}/figuras", exist_ok=True); os.makedirs(f"{OUT}/tabelas", exist_ok=True)

# (opcional) salvar também no Google Drive
SALVAR_NO_DRIVE = False
if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/tcc_drift/resultados"
    os.makedirs(f"{OUT}/figuras", exist_ok=True); os.makedirs(f"{OUT}/tabelas", exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, time, warnings
from scipy import stats
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

# ---- Parâmetros do gerador -------------------------------------------------
N_FEAT   = 4                              # x0,x1 = subespaço de drift; x2,x3 estáveis
W_TRUE   = np.array([1.5, 1.0, 1.0, 0.5]) # pesos do conceito y = 1[w·(x-mu_t)+ruído>0]
NOISE    = 0.5
B        = 400                            # tamanho do bloco (instâncias)
N_BLOCKS = 40
N        = B * N_BLOCKS                   # 16 000 instâncias por stream
ONSET    = 6400                           # início do drift (bloco 16)
TRANS    = 2400                           # largura da transição gradual/incremental (6 blocos)
INC_STEP = 100                            # tamanho do passo do drift incremental
REC_SEG  = 4000                           # duração de cada estado no recorrente (10 blocos)
K_SLIDE  = 4                              # cadência da janela deslizante (a cada 4 blocos)
K_EP     = 8                              # blocos observados após uma detecção (sinais derivados)
HDDDM_STAT  = "stdev"                     # variante do limiar adaptativo: "stdev" (gamma) ou "tstat" (alpha)
HDDDM_GAMMA = 2.0
ALPHA       = 0.05

SEVERITY   = {"baixa": 0.1, "media": 0.4, "alta": 0.7}
DRIFT_TYPES = ["abrupto", "gradual", "incremental", "recorrente"]
CLASSIFIERS = ["NaiveBayes", "RandomForest", "XGBoost"]
STRATEGIES  = ["reset", "janela", "incremental", "reativacao", "controle"]

import xgboost, sklearn, scipy
print("python", platform.python_version(), "| numpy", np.__version__, "| sklearn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| scipy", scipy.__version__)

# Paleta (estratégias em ordem fixa; controle em cinza neutro)
COR = {"reset": "#2a78d6", "janela": "#eb6834", "incremental": "#1baf7a",
       "reativacao": "#eda100", "controle": "#8a8a85"}
ROTULO = {"reset": "Reset completo", "janela": "Janela deslizante", "incremental": "Incremental",
          "reativacao": "Reativação", "controle": "Sem retreinamento"}
COR_TIPO = {"abrupto": "#2a78d6", "gradual": "#eb6834", "incremental": "#1baf7a", "recorrente": "#e87ba4"}
mpl.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 10, "legend.frameon": False})
def salvar(fig, nome):
    fig.savefig(f"{OUT}/figuras/{nome}.pdf"); fig.savefig(f"{OUT}/figuras/{nome}.png", dpi=200)

## 1. Severidade: forma fechada da distância de Hellinger (Seção 1.4)

Para $P=\mathcal N(\mu,\sigma^2)$ e $Q=\mathcal N(\mu+\Delta,\sigma^2)$: $H^2 = 1-\exp(-\Delta^2/8\sigma^2)$ (Eq. 1.2), invertível em $\Delta/\sigma = \sqrt{-8\ln(1-H^2_{alvo})}$ (Eq. 1.3).
No caso multivariado com covariância $\sigma^2 I$ a expressão vale com $\|\delta\|^2$ no lugar de $\Delta^2$, o que permite construir o padrão recorrente com três estados equidistantes (triângulo equilátero no plano $x_0, x_1$) — cada transição tem exatamente a mesma severidade.

In [ ]:
def h2_normais(mu_p, mu_q, s_p=1.0, s_q=1.0):
    """Eq. 1.1 – H² exata entre N(mu_p, s_p²) e N(mu_q, s_q²)."""
    return 1 - np.sqrt(2*s_p*s_q/(s_p**2+s_q**2)) * np.exp(-(mu_p-mu_q)**2/(4*(s_p**2+s_q**2)))

def delta_de_h2(h2_alvo, sigma=1.0):
    """Eq. 1.3 – deslocamento Δ que produz a severidade-alvo."""
    return sigma*np.sqrt(-8*np.log(1-h2_alvo))

def h2_real_multivariada(delta_vec, sigma=1.0):
    """H² exata entre N(mu, σ²I) e N(mu+δ, σ²I): 1-exp(-||δ||²/8σ²) (generaliza a Eq. 1.2)."""
    return 1 - np.exp(-np.sum(np.asarray(delta_vec)**2)/(8*sigma**2))

tab2 = pd.DataFrame([(n, h, delta_de_h2(h), h2_normais(0, delta_de_h2(h))) for n, h in SEVERITY.items()],
                    columns=["Nível", "H2_alvo", "Delta/sigma", "H2 (Eq. 1.1, conferência)"])
display(tab2.round(3))
assert np.allclose(tab2["H2_alvo"], tab2["H2 (Eq. 1.1, conferência)"])

## 2. Geração da stream sintética (Seção 2.2)

* 4 atributos independentes $x \sim \mathcal N(\mu_t, I)$; o drift ocorre no subespaço $(x_0, x_1)$.
* Rótulo $y = \mathbb 1[w^\top(x-\mu_t) + \varepsilon > 0]$, $w=(1{,}5;\,1;\,1;\,0{,}5)$, $\varepsilon\sim\mathcal N(0, 0{,}5^2)$: a fronteira de decisão acompanha os dados, de modo que o deslocamento de $p(X)$ também altera $p(y\mid X)$ e degrada o modelo antigo proporcionalmente à severidade.
* Stream de 16 000 instâncias em blocos de 400. Drift a partir da instância 6 400 (abrupto); transição de 2 400 instâncias (gradual e incremental, passos de 100); recorrente $P\to Q\to R\to P$ com estados de 4 000 instâncias.

In [ ]:
def gerar_stream(tipo, h2_alvo, seed):
    """Retorna X, y, mu (média geradora por instância) e lista de transições."""
    rng = np.random.default_rng(seed)
    d = delta_de_h2(h2_alvo)
    muP = np.zeros(N_FEAT)
    muQ = muP.copy(); muQ[0] = d
    muR = muP.copy(); muR[0] = d/2; muR[1] = d*np.sqrt(3)/2   # triângulo equilátero: ||P-Q||=||Q-R||=||R-P||=Δ
    t = np.arange(N)
    mu = np.zeros((N, N_FEAT))
    if tipo == "abrupto":
        mu[t >= ONSET] = muQ
        trans = [dict(ini=ONSET, fim=ONSET, recorrencia=False)]
    elif tipo == "gradual":
        p = np.clip((t-ONSET)/TRANS, 0, 1)
        deQ = rng.random(N) < p
        mu[deQ] = muQ
        trans = [dict(ini=ONSET, fim=ONSET+TRANS, recorrencia=False)]
    elif tipo == "incremental":
        frac = np.where(t >= ONSET, np.minimum(1, (np.floor((t-ONSET)/INC_STEP)+1)*INC_STEP/TRANS), 0)
        mu = frac[:, None]*muQ[None, :]
        trans = [dict(ini=ONSET, fim=ONSET+TRANS, recorrencia=False)]
    elif tipo == "recorrente":
        estados = [muP, muQ, muR, muP]
        for s in range(4):
            mu[s*REC_SEG:(s+1)*REC_SEG] = estados[s]
        trans = [dict(ini=REC_SEG*k, fim=REC_SEG*k, recorrencia=(k == 3)) for k in (1, 2, 3)]
    else:
        raise ValueError(tipo)
    Z = rng.normal(size=(N, N_FEAT))
    X = Z + mu
    y = ((Z @ W_TRUE + NOISE*rng.normal(size=N)) > 0).astype(int)  # conceito acompanha mu_t (real + virtual)
    return X, y, mu, trans

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(13, 3.0), sharey=True)
idx = np.arange(0, N, 16)
for ax, tipo in zip(axs, DRIFT_TYPES):
    X, y, mu, tr = gerar_stream(tipo, SEVERITY["media"], seed=7)
    d = delta_de_h2(SEVERITY["media"])
    if tipo == "gradual":
        # cada instância vem de P ou de Q; a linha é a média esperada p(t)·Δ
        deQ = mu[idx, 0] > 0
        ax.scatter(idx[~deQ], X[idx[~deQ], 0], s=3, alpha=0.4, color="#8a8a85", linewidths=0, label="de P")
        ax.scatter(idx[deQ], X[idx[deQ], 0], s=3, alpha=0.4, color=COR_TIPO[tipo], linewidths=0, label="de Q")
        esperado = np.clip((np.arange(N)-ONSET)/TRANS, 0, 1)*d
        ax.plot(np.arange(N), esperado, color="black", lw=1.5, label="média esperada")
        ax.legend(fontsize=7, loc="upper left", markerscale=3)
    else:
        ax.scatter(idx, X[idx, 0], s=3, alpha=0.35, color=COR_TIPO[tipo], linewidths=0)
        ax.plot(np.arange(N), mu[:, 0], color="black", lw=1.5, label="$\\mu_{x_0}$")
    if tipo == "recorrente":
        # R difere de P e de Q também em x1 (triângulo equilátero no plano x0-x1)
        ax.plot(np.arange(N), mu[:, 1], color="black", lw=1.2, ls="--", label="$\\mu_{x_1}$")
        for k, nome in enumerate("PQRP"):
            ax.text((k+0.5)*REC_SEG, 4.3, nome, ha="center", fontsize=10, fontweight="bold")
        ax.legend(fontsize=7, loc="lower right")
    for t in tr:
        ax.axvline(t["ini"], color="gray", ls=":", lw=1)
        if t["fim"] != t["ini"]: ax.axvline(t["fim"], color="gray", ls=":", lw=1)
    ax.set_title(tipo.capitalize()); ax.set_xlabel("instância")
axs[0].set_ylabel("$x_0$"); axs[0].set_ylim(-3.5, 5)
fig.suptitle("Padrões temporais gerados (severidade média, $\\Delta/\\sigma \\approx 2{,}02$)", y=1.04)
salvar(fig, "fig_padroes_drift"); plt.show()

Gradual e incremental têm a **mesma média esperada** ao longo da transição; o que os distingue é a forma da distribuição. No gradual cada instância vem inteira de $P$ ou de $Q$ (mistura bimodal); no incremental há uma única normal cujo centro se desloca.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
bins = np.linspace(-4, 7, 45); d = delta_de_h2(SEVERITY["alta"])
for ax, frac in zip(axs, (0.25, 0.5, 0.75)):
    c = int(ONSET + TRANS*frac)
    for tipo in ("gradual", "incremental"):
        v = np.concatenate([gerar_stream(tipo, SEVERITY["alta"], s)[0][c-200:c+200, 0] for s in range(30)])
        ax.hist(v, bins=bins, density=True, histtype="step", lw=2, color=COR_TIPO[tipo], label=tipo)
    ax.axvline(0, color="gray", ls=":", lw=1); ax.axvline(d, color="gray", ls=":", lw=1)
    ax.text(-0.35, 0.43, "$\\mu_P$", ha="center", color="gray"); ax.text(d+0.35, 0.43, "$\\mu_Q$", ha="center", color="gray")
    ax.set_title(f"{int(frac*100)}% da transição"); ax.set_xlabel("$x_0$")
axs[0].set_ylabel("densidade"); axs[0].legend()
fig.suptitle("Mesma média, distribuições diferentes: $x_0$ durante a transição (severidade alta)", y=1.03)
salvar(fig, "fig_gradual_vs_incremental"); plt.show()

## 3. Distância de Hellinger amostral e HDDDM (Seção 1.3)

O HDDDM compara, a cada bloco, a referência acumulada desde a última mudança com o bloco novo (histogramas com $\lfloor\sqrt n\rfloor$ bins por atributo). A **severidade observada** é a mesma grandeza na escala $H^2$: como os atributos são independentes, o coeficiente de Bhattacharyya fatora e $\hat H^2 = 1-\prod_k \widehat{BC}_k$.

O limiar adaptativo segue Ditzler e Polikar (2011), que propõem duas variantes; adotamos a baseada em desvio-padrão ($\beta = \hat\varepsilon + \gamma\hat\sigma$, $\gamma=2$) porque a variante pela estatística *t* ($\hat\sigma/\sqrt{k}$) tem taxa de falso alarme crescente com o tempo de estabilidade (diagnóstico na célula a seguir).

In [ ]:
def bc_por_feature(A, Bm, bins=None):
    """Coeficiente de Bhattacharyya por feature via histogramas com bordas comuns."""
    if bins is None:
        bins = int(np.floor(np.sqrt(len(Bm))))   # regra de Ditzler & Polikar (2011)
    bc = np.empty(A.shape[1])
    for k in range(A.shape[1]):
        lo = min(A[:, k].min(), Bm[:, k].min()); hi = max(A[:, k].max(), Bm[:, k].max())
        e = np.linspace(lo, hi, bins+1)
        pa = np.histogram(A[:, k], e)[0].astype(float); pb = np.histogram(Bm[:, k], e)[0].astype(float)
        bc[k] = np.sum(np.sqrt(pa/pa.sum()*pb/pb.sum()))
    return np.clip(bc, 0, 1)

def h2_observada(A, Bm, bins=None):
    """Severidade observada: H² conjunta = 1-∏BC_k (válida p/ features independentes)."""
    return 1 - np.prod(bc_por_feature(A, Bm, bins))

class HDDDM:
    """HDDDM (Ditzler & Polikar, 2011): distância de Hellinger média por feature entre a
    referência (acumulada desde a última mudança) e o bloco novo; limiar adaptativo
    limiar adaptativo nas duas variantes do método:
      "stdev": beta = eps_hat + gamma * sigma_hat
      "tstat": beta = eps_hat + t_{alpha/2,k-1} * sigma_hat / sqrt(k)"""
    def __init__(self, X_ref, stat=None, gamma=None, alpha=None):
        self.ref = X_ref.copy()
        self.stat = stat or HDDDM_STAT; self.gamma = gamma or HDDDM_GAMMA; self.alpha = alpha or ALPHA
        self.prev = None; self.eps = []

    def update(self, Xb):
        bc = bc_por_feature(self.ref, Xb)
        delta = np.mean(np.sqrt(np.maximum(0, 2-2*bc)))   # Hellinger de Ditzler (escala [0, √2])
        h2 = 1 - np.prod(bc)                                 # mesma grandeza em escala H² (Eq. 1.2)
        drift = False
        if self.prev is not None:
            e = abs(delta-self.prev)
            if len(self.eps) >= 2:
                k = len(self.eps)
                m, sd = np.mean(self.eps), np.std(self.eps, ddof=1)
                if self.stat == "tstat":
                    beta = m + stats.t.ppf(1-self.alpha/2, k-1)*sd/np.sqrt(k)
                else:
                    beta = m + self.gamma*sd
                drift = e > beta
            if not drift:
                self.eps.append(e)
        ref_antes = self.ref
        if drift:
            self.ref = Xb.copy(); self.prev = None; self.eps = []
        else:
            self.ref = np.vstack([self.ref, Xb]); self.prev = delta
        return drift, h2, ref_antes


def rodar_detector(X, gamma=None):
    """Executa o HDDDM sobre a stream (independe do classificador e da estratégia)."""
    det = HDDDM(X[:B], gamma=gamma)
    eventos = np.zeros(N_BLOCKS, bool); h2_obs = np.full(N_BLOCKS, np.nan); refs = {}
    for b in range(1, N_BLOCKS):
        drift, h2, ref_antes = det.update(X[b*B:(b+1)*B])
        eventos[b] = drift; h2_obs[b] = h2
        if drift:
            refs[b] = ref_antes
    return eventos, h2_obs, refs

### 3.1 Validação: a severidade observada converge para a real (cronograma, 31/08–20/09)

In [ ]:
rng = np.random.default_rng(0)
ns = [100, 200, 400, 800, 1600, 3200, 6400, 12800]
linhas = []
for nivel, h in SEVERITY.items():
    d = delta_de_h2(h)
    for n in ns:
        v = [h2_observada(rng.normal(size=(n, N_FEAT)), rng.normal(size=(n, N_FEAT)) + np.r_[d, 0, 0, 0]) for _ in range(60)]
        linhas.append(dict(nivel=nivel, h2_real=h, n=n, h2_obs=np.mean(v), dp=np.std(v)))
for n in ns:
    v = [h2_observada(rng.normal(size=(n, N_FEAT)), rng.normal(size=(n, N_FEAT))) for _ in range(60)]
    linhas.append(dict(nivel="sem drift", h2_real=0.0, n=n, h2_obs=np.mean(v), dp=np.std(v)))
conv = pd.DataFrame(linhas)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for (nivel, g), c in zip(conv.groupby("nivel", sort=False), ["#2a78d6", "#eb6834", "#1baf7a", "#8a8a85"]):
    ax.errorbar(g.n, g.h2_obs, yerr=g.dp, marker="o", ms=4, lw=1.5, capsize=2, color=c, label=f"{nivel}")
    ax.axhline(g.h2_real.iloc[0], color=c, ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("tamanho da janela (instâncias)"); ax.set_ylabel(r"$\hat H^2$ observada")
ax.set_title("Severidade observada → severidade real (tracejado)"); ax.legend(ncol=2, fontsize=8)
salvar(fig, "fig_convergencia_severidade"); plt.show()
display(conv.pivot(index="n", columns="nivel", values="h2_obs").round(3))

### 3.2 Diagnóstico das duas variantes do limiar do HDDDM

In [ ]:
def diag_detector(stat, par, reps=10):
    fp = nb = 0; hits = {t: [] for t in DRIFT_TYPES}
    for tipo in DRIFT_TYPES:
        for h in SEVERITY.values():
            for r in range(reps):
                X, *_ , tr = gerar_stream(tipo, h, 90_000 + r)
                d = HDDDM(X[:B], stat=stat, gamma=par, alpha=par)
                ev = np.array([False] + [d.update(X[b*B:(b+1)*B])[0] for b in range(1, N_BLOCKS)])
                b0 = tr[0]["ini"]//B; fp += ev[1:b0].sum(); nb += b0-1
                for t in tr:
                    hits[tipo].append(ev[t["ini"]//B: int(np.ceil(t["fim"]/B))+4].any())
    return dict(variante=stat, parametro=par, falso_alarme_por_bloco=fp/nb, **{f"det_{k}": np.mean(v) for k, v in hits.items()})
diag = pd.DataFrame([diag_detector("tstat", 0.05), diag_detector("tstat", 0.01),
                     diag_detector("stdev", 1.5), diag_detector("stdev", 2.0), diag_detector("stdev", 2.5)])
display(diag.round(3)); diag.to_csv(f"{OUT}/diagnostico_hdddm.csv", index=False)

## 4. Classificadores e adaptação a stream (Seção 2.3)

In [ ]:
class NB:
    def __init__(self, seed): self.seed = seed
    def fit(self, X, y): self.m = GaussianNB().fit(X, y); return self
    def update(self, X, y): self.m.partial_fit(X, y); return self      # contadores nativos
    def predict(self, X): return self.m.predict(X)

class RF:
    """Floresta de árvores substituíveis: reset/janela = nova floresta; incremental = novas
    árvores adicionadas sem descartar as antigas."""
    N_INIT, N_ADD = 50, 25
    def __init__(self, seed): self.seed = seed; self.k = 0
    def _forest(self, n, X, y):
        self.k += 1
        return RandomForestClassifier(n_estimators=n, max_features="sqrt", n_jobs=1,
                                      random_state=self.seed*1000+self.k).fit(X, y)
    def fit(self, X, y): self.fs = [self._forest(self.N_INIT, X, y)]; return self
    def update(self, X, y): self.fs.append(self._forest(self.N_ADD, X, y)); return self
    def predict(self, X):
        n = np.array([f.n_estimators for f in self.fs], float)
        p = sum(ni*f.predict_proba(X)[:, 1] for ni, f in zip(n, self.fs))/n.sum()
        return (p > 0.5).astype(int)

class XGB:
    """Boosting: reset/janela = novo conjunto; incremental = novas árvores em micro-lote
    corrigindo o conjunto já construído (xgb_model)."""
    N_INIT, N_ADD = 50, 25
    def __init__(self, seed): self.seed = seed
    def _new(self, n):
        return XGBClassifier(n_estimators=n, max_depth=4, learning_rate=0.3, subsample=1.0,
                             tree_method="hist", n_jobs=1, random_state=self.seed, verbosity=0)
    def fit(self, X, y): self.m = self._new(self.N_INIT).fit(X, y); return self
    def update(self, X, y):
        m = self._new(self.N_ADD); m.fit(X, y, xgb_model=self.m.get_booster()); self.m = m; return self
    def predict(self, X): return self.m.predict(X)

CLF = {"NaiveBayes": NB, "RandomForest": RF, "XGBoost": XGB}

## 5. Estratégias de retreinamento (Seção 2.4)

Avaliação *prequential* (teste-então-treino) por bloco: como os modelos permanecem congelados entre atualizações, a acurácia do bloco é idêntica à prequential instância a instância. Todas as estratégias usam como janela de treino o bloco mais recente (400 instâncias); a janela deslizante retreina a cada `K_SLIDE = 4` blocos.

Na reativação, quando a menor distância ao repositório não fica abaixo do limiar $\theta$ (nenhuma recorrência), o modelo é treinado do zero, como no reset.

In [ ]:
def rodar_estrategia(X, y, eventos, clf_name, estrategia, seed, theta_rec, k_slide=None):
    """Prequential por bloco (teste-então-treino): o modelo fica congelado dentro do bloco,
    portanto acurácia por bloco = prequential instância a instância."""
    Mk = CLF[clf_name]; k_slide = k_slide or K_SLIDE
    X0, y0 = X[:B], y[:B]
    model = Mk(seed).fit(X0, y0)
    train_X = X0
    repo = []
    acc = np.full(N_BLOCKS, np.nan)
    n_treinos = n_reativ = 0; t_treino = 0.0
    for b in range(1, N_BLOCKS):
        Xb, yb = X[b*B:(b+1)*B], y[b*B:(b+1)*B]
        acc[b] = np.mean(model.predict(Xb) == yb)        # teste
        t0 = time.perf_counter(); treinou = False
        if estrategia == "reset" and eventos[b]:
            model = Mk(seed+b).fit(Xb, yb); train_X = Xb; treinou = True
        elif estrategia == "janela" and b % k_slide == 0:
            model = Mk(seed+b).fit(Xb, yb); train_X = Xb; treinou = True
        elif estrategia == "incremental" and eventos[b]:
            model.update(Xb, yb); treinou = True
        elif estrategia == "reativacao" and eventos[b]:
            repo.append((model, train_X))
            dist = [h2_observada(tx, Xb) for _, tx in repo]
            j = int(np.argmin(dist))
            if dist[j] < theta_rec:
                model, train_X = repo.pop(j); n_reativ += 1
            else:                                       # sem recorrência: treina do zero
                model = Mk(seed+b).fit(Xb, yb); train_X = Xb; treinou = True
        if treinou:
            n_treinos += 1; t_treino += time.perf_counter()-t0
    return acc, n_treinos, n_reativ, t_treino

In [ ]:
TOL_REC = 0.03
def metricas(acc, trans, tipo):
    b_on = trans[0]["ini"]//B
    pre = np.nanmean(acc[1:b_on])
    pos = np.nanmean(acc[b_on:])
    recs, ok = [], []
    for i, tr in enumerate(trans):
        b_fim = int(np.ceil(tr["fim"]/B))
        b_lim = trans[i+1]["ini"]//B if i+1 < len(trans) else N_BLOCKS
        seg = acc[b_fim:b_lim]
        hit = np.where(seg >= pre-TOL_REC)[0]
        if len(hit):
            recs.append(hit[0]*B); ok.append(True)
        else:
            recs.append((b_lim-b_fim)*B); ok.append(False)     # censurado
    return dict(acc_pre=pre, acc_pos=pos, acc_total=np.nanmean(acc[1:]),
                queda_min=pre-np.nanmin(acc[b_on:]),
                t_recuperacao=np.mean(recs), recuperou=np.mean(ok))

In [ ]:
def sinais_detector(X, eventos, refs, theta_rec):
    """Para cada episódio (1ª detecção + K_EP blocos), calcula os três sinais observáveis
    usando somente o que estaria disponível em produção (a referência no momento da
    detecção e os blocos seguintes)."""
    eps_list, repo, ult = [], [], -10**9
    for b in np.where(eventos)[0]:
        if b < ult + K_EP:
            continue
        ult = b
        R = refs[b]
        js = range(0, min(K_EP, N_BLOCKS-b))
        T = np.array([h2_observada(R, X[(b+j)*B:(b+j+1)*B]) for j in js])
        plato = X[(b+js[-1])*B:(b+js[-1]+1)*B]
        s_sev = float(np.median(T[-3:]))
        s_vel = float(T[0]/max(T.max(), 1e-9))
        rise = int(np.argmax(T >= 0.9*T.max()))
        s_rec = float(min([h2_observada(r, plato) for r in repo])) if repo else np.inf
        eps_list.append(dict(bloco=int(b), s_sev=s_sev, s_vel=s_vel, rise=rise, s_rec=s_rec,
                             flag_rec=s_rec < theta_rec, traj=T))
        repo.append(R)
    return eps_list


def casar_episodios(eps_list, trans, tol_blocos=3):
    """Associa cada transição verdadeira ao 1º episódio que começa em [ini, fim+tol]."""
    out = []
    for i, tr in enumerate(trans):
        bi, bf = tr["ini"]//B, int(np.ceil(tr["fim"]/B)) + tol_blocos
        cand = [e for e in eps_list if bi <= e["bloco"] <= bf]
        out.append((i, tr, cand[0] if cand else None))
    return out

### 5.1 Calibração do limiar de recorrência $\theta$

$\theta$ é o percentil 99 de $\hat H^2$ entre pares de blocos **da mesma distribuição**, estimado apenas com trechos estacionários (pré-drift) — informação disponível em produção, sem uso do *ground truth* de severidade.

In [ ]:
null = []
for r in range(30):
    X, *_ = gerar_stream("abrupto", 0.1, 50_000 + r)
    for i in range(1, ONSET//B - 2, 2):
        null.append(h2_observada(X[i*B:(i+1)*B], X[(i+1)*B:(i+2)*B]))
THETA_REC = float(np.percentile(null, 99))
print(f"theta_rec = {THETA_REC:.4f}  (média nula = {np.mean(null):.4f}, n = {len(null)} pares)")

## 6. Execução do experimento fatorial

In [ ]:
def executar_stream(tipo, nivel, rep):
    ti, si = DRIFT_TYPES.index(tipo), list(SEVERITY).index(nivel)
    seed = 1000*rep + 10*ti + si
    h2 = SEVERITY[nivel]
    X, y, mu, trans = gerar_stream(tipo, h2, seed)
    eventos, h2_obs, refs = rodar_detector(X)
    eps = sinais_detector(X, eventos, refs, THETA_REC)
    base = dict(tipo=tipo, severidade=nivel, h2_real=h2, rep=rep, seed=seed)

    linhas, curvas = [], []
    for clf in CLASSIFIERS:
        for est in STRATEGIES:
            acc, nt, nr, tt = rodar_estrategia(X, y, eventos, clf, est, seed, THETA_REC)
            m = metricas(acc, trans, tipo)
            linhas.append(dict(**base, classificador=clf, estrategia=est, **m, n_treinos=nt,
                               n_reativacoes=nr, tempo_treino_s=tt,
                               taxa_acionamento=(nt+nr)/(N_BLOCKS-1), n_deteccoes=int(eventos.sum())))
            curvas.append(dict(**base, classificador=clf, estrategia=est, acc=acc))

    b0 = trans[0]["ini"]//B
    det = dict(**base, n_deteccoes=int(eventos.sum()), falsos_alarmes_pre=int(eventos[1:b0].sum()))
    casados = []
    for i, tr, e in casar_episodios(eps, trans):
        casados.append(dict(**base, transicao=i, recorrencia=tr["recorrencia"], detectado=e is not None,
                            atraso_blocos=(e["bloco"] - tr["ini"]//B) if e else np.nan,
                            h2_deteccao=h2_obs[e["bloco"]] if e else np.nan,
                            **({k: e[k] for k in ("s_sev", "s_vel", "rise", "s_rec", "flag_rec")} if e else {})))
    todos = [dict(**base, **{k: e[k] for k in ("bloco", "s_sev", "s_vel", "rise", "s_rec", "flag_rec")}) for e in eps]
    return linhas, curvas, det, casados, todos

tarefas = [(t, s, r) for t in DRIFT_TYPES for s in SEVERITY for r in range(N_REPS)]
t0 = time.time()
saida = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(executar_stream)(*a) for a in tarefas)
print(f"{len(tarefas)} streams × 15 execuções = {len(tarefas)*15} execuções em {time.time()-t0:.0f}s")

res     = pd.DataFrame([l for s in saida for l in s[0]])
curvas  = [c for s in saida for c in s[1]]
det     = pd.DataFrame([s[2] for s in saida])
eps_cas = pd.DataFrame([c for s in saida for c in s[3]])
eps_all = pd.DataFrame([c for s in saida for c in s[4]])
assert res.groupby(["tipo", "severidade", "classificador", "estrategia"]).ngroups == 180

res.to_csv(f"{OUT}/resultados_brutos.csv", index=False)
det.to_csv(f"{OUT}/deteccao_por_stream.csv", index=False)
eps_cas.to_csv(f"{OUT}/sinais_por_transicao.csv", index=False)
eps_all.to_csv(f"{OUT}/sinais_todos_episodios.csv", index=False)
pd.DataFrame([dict(**{k: v for k, v in c.items() if k != "acc"}, bloco=b, acc=a)
              for c in curvas for b, a in enumerate(c["acc"]) if not np.isnan(a)]
             ).to_csv(f"{OUT}/curvas_prequential.csv.gz", index=False)
res.head()

### 6.1 Comportamento do detector (fixo em todas as condições)

In [ ]:
d1 = eps_cas.groupby(["tipo", "severidade"], sort=False).agg(
    taxa_deteccao=("detectado", "mean"), atraso_medio_blocos=("atraso_blocos", "mean")).reset_index()
d2 = det.groupby(["tipo", "severidade"], sort=False).agg(
    deteccoes_por_stream=("n_deteccoes", "mean"), falsos_alarmes_pre=("falsos_alarmes_pre", "mean")).reset_index()
tab_det = d1.merge(d2, on=["tipo", "severidade"])
display(tab_det.round(2))

## 7. Resultados – Questão principal

In [ ]:
ORD_T, ORD_S, ORD_E = DRIFT_TYPES, list(SEVERITY), STRATEGIES
res["tipo"] = pd.Categorical(res.tipo, ORD_T, ordered=True)
res["severidade"] = pd.Categorical(res.severidade, ORD_S, ordered=True)
res["estrategia"] = pd.Categorical(res.estrategia, ORD_E, ordered=True)
ctrl = res[res.estrategia == "controle"].set_index(["tipo", "severidade", "classificador", "rep"])["acc_pos"]
res["ganho_vs_controle"] = res.acc_pos.values - ctrl.loc[list(zip(res.tipo, res.severidade, res.classificador, res.rep))].values

media = res.groupby(["classificador", "tipo", "severidade", "estrategia"], observed=True).agg(
    acc_pos=("acc_pos", "mean"), dp=("acc_pos", "std"), ganho=("ganho_vs_controle", "mean"),
    t_rec=("t_recuperacao", "mean"), recuperou=("recuperou", "mean"),
    acionamento=("taxa_acionamento", "mean"), tempo_treino=("tempo_treino_s", "mean")).reset_index()
media.to_csv(f"{OUT}/resultados_por_condicao.csv", index=False)

melhor = (media[media.estrategia != "controle"].sort_values("acc_pos", ascending=False)
          .groupby(["classificador", "tipo", "severidade"], observed=True).head(1)
          .sort_values(["classificador", "tipo", "severidade"]))
print("Melhor estratégia (acurácia prequential pós-drift média) por célula:")
display(melhor.pivot_table(index=["tipo", "severidade"], columns="classificador", values="estrategia", aggfunc="first", observed=True))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 6.2), sharey=True)
for ax, clf in zip(axs, CLASSIFIERS):
    m = media[media.classificador == clf].pivot_table(index=["tipo", "severidade"], columns="estrategia",
                                                    values="acc_pos", observed=True)[ORD_E]
    im = ax.imshow(m.values, cmap="Blues", vmin=0.5, vmax=0.95, aspect="auto")
    ax.set_xticks(range(len(ORD_E))); ax.set_xticklabels([ROTULO[e] for e in ORD_E], rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels([f"{t} · {s}" for t, s in m.index], fontsize=8)
    for i in range(m.shape[0]):
        best = m.values[i, :-1].max()
        for j in range(m.shape[1]):
            v = m.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if v > 0.78 else "black", fontweight="bold" if (j < m.shape[1]-1 and v == best) else "normal")
    for k in (3, 6, 9): ax.axhline(k-0.5, color="white", lw=2)
    ax.set_title(clf); ax.grid(False)
fig.colorbar(im, ax=axs, shrink=0.7, label="acurácia prequential pós-drift")
fig.suptitle("Eficácia por estratégia, tipo e severidade (negrito = melhor estratégia da linha)", y=0.98)
salvar(fig, "fig_heatmap_eficacia"); plt.show()

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(14, 8), sharex=True, sharey=True)
for i, clf in enumerate(CLASSIFIERS):
    for j, tipo in enumerate(DRIFT_TYPES):
        ax = axs[i, j]
        for est in STRATEGIES:
            A = np.array([c["acc"] for c in curvas if c["classificador"] == clf and c["tipo"] == tipo
                          and c["severidade"] == "media" and c["estrategia"] == est])
            ax.plot(np.arange(N_BLOCKS)*B, np.nanmean(A, 0), color=COR[est], lw=1.6 if est != "controle" else 1.2,
                    ls="-" if est != "controle" else "--", label=ROTULO[est])
        _, _, _, tr = gerar_stream(tipo, 0.4, 0)
        for t in tr: ax.axvspan(t["ini"], max(t["fim"], t["ini"]+60), color="gray", alpha=0.15, lw=0)
        if i == 0: ax.set_title(tipo.capitalize())
        if j == 0: ax.set_ylabel(f"{clf}\nacurácia do bloco")
        if i == 2: ax.set_xlabel("instância")
axs[0, 0].set_ylim(0.45, 0.97)
axs[0, -1].legend(loc="lower left", fontsize=7)
fig.suptitle("Acurácia prequential média por bloco – severidade média (faixa cinza = transição)", y=1.0)
salvar(fig, "fig_curvas_prequential"); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
g = res[res.estrategia != "controle"].groupby(["estrategia", "severidade"], observed=True)["ganho_vs_controle"].mean().unstack()
x = np.arange(len(ORD_S)); w = 0.2
for k, est in enumerate(ORD_E[:-1]):
    axs[0].bar(x + (k-1.5)*w, g.loc[est], w*0.9, color=COR[est], label=ROTULO[est])
axs[0].set_xticks(x); axs[0].set_xticklabels(ORD_S); axs[0].set_ylabel("ganho de acurácia vs. controle")
axs[0].set_title("Ganho sobre o controle por severidade"); axs[0].legend(fontsize=7)
t = res[res.estrategia != "controle"].groupby(["estrategia", "tipo"], observed=True)["t_recuperacao"].mean().unstack()
x = np.arange(len(ORD_T))
for k, est in enumerate(ORD_E[:-1]):
    axs[1].bar(x + (k-1.5)*w, t.loc[est], w*0.9, color=COR[est])
axs[1].set_xticks(x); axs[1].set_xticklabels(ORD_T); axs[1].set_ylabel("instâncias até recuperar")
axs[1].set_title("Tempo de recuperação")
c = res.groupby("estrategia", observed=True)[["taxa_acionamento", "tempo_treino_s"]].mean().loc[ORD_E]
axs[2].bar([ROTULO[e] for e in ORD_E], c.taxa_acionamento, color=[COR[e] for e in ORD_E])
axs[2].set_ylabel("acionamentos por bloco"); axs[2].set_title("Custo: taxa de acionamento")
axs[2].tick_params(axis="x", rotation=25, labelsize=8)
salvar(fig, "fig_ganho_recuperacao_custo"); plt.show()
display(res.groupby(["classificador", "estrategia"], observed=True)[["taxa_acionamento", "n_treinos", "tempo_treino_s"]].mean().round(3))

### 7.1 Análise estatística

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

mod = smf.ols("acc_pos ~ C(tipo) * C(severidade) * C(classificador) * C(estrategia)", data=res.astype({"tipo": str, "severidade": str, "estrategia": str})).fit()
an = anova_lm(mod, typ=2)
an["eta2_parcial"] = an["sum_sq"] / (an["sum_sq"] + an.loc["Residual", "sum_sq"])
an.index = an.index.str.replace("C(", "", regex=False).str.replace(")", "", regex=False)
an_show = an.drop("Residual").sort_values("eta2_parcial", ascending=False)
print(f"ANOVA fatorial completa (R² = {mod.rsquared:.3f}, n = {len(res)})")
display(an_show[["df", "F", "PR(>F)", "eta2_parcial"]].round(4))

In [ ]:
wide = res.pivot_table(index=["tipo", "severidade", "classificador", "rep"], columns="estrategia",
                       values="acc_pos", observed=True)[ORD_E]
fr = stats.friedmanchisquare(*[wide[e] for e in ORD_E])
ranks = wide.rank(axis=1, ascending=False).mean().round(2)
print(f"Friedman (n = {len(wide)} blocos): chi2 = {fr.statistic:.1f}, p = {fr.pvalue:.2e}")
print("Rank médio (1 = melhor):", ranks.to_dict())

pares = [(a, b) for i, a in enumerate(ORD_E) for b in ORD_E[i+1:]]
p = [stats.wilcoxon(wide[a], wide[b]).pvalue for a, b in pares]
rej, p_holm, *_ = multipletests(p, method="holm")
wil = pd.DataFrame([dict(A=a, B=b, mediana_dif=float(np.median(wide[a]-wide[b])), p_holm=ph, significativo=r)
                    for (a, b), ph, r in zip(pares, p_holm, rej)])
display(wil.round(4))

# Friedman por tipo de drift × classificador
linhas = []
for (tipo, clf), g in wide.groupby(level=["tipo", "classificador"], observed=True):
    f = stats.friedmanchisquare(*[g[e] for e in ORD_E]); r = g.rank(axis=1, ascending=False).mean()
    linhas.append(dict(tipo=tipo, classificador=clf, chi2=f.statistic, p=f.pvalue, melhor_rank=r.idxmin(), **r.round(2).to_dict()))
fr_tc = pd.DataFrame(linhas); display(fr_tc.round(4))

## 8. Questão derivada – sinais observáveis do detector (Seção 2.7)

Para cada transição verdadeira, toma-se o primeiro episódio de detecção que começa entre o início da transição e 3 blocos após seu fim. Os três sinais usam apenas a referência do detector no instante da detecção e os `K_EP = 8` blocos seguintes:

* **Severidade observada** $S_{sev}$: mediana de $\hat H^2(\text{referência}, \text{bloco})$ nos 3 últimos blocos do episódio (patamar).
* **Velocidade** $S_{vel}$: fração da divergência final já presente no bloco da detecção ($\approx 1$ para mudanças abruptas, $<1$ para transições lentas).
* **Recorrência** $S_{rec}$: menor $\hat H^2$ entre o patamar e as referências de episódios anteriores; sinaliza recorrência se $S_{rec} < \theta$.

In [ ]:
from sklearn.metrics import roc_auc_score
e = eps_cas[eps_cas.detectado].copy()

# (a) severidade observada vs real
sev = e.groupby("severidade", sort=False).agg(h2_real=("h2_real", "first"), s_sev_media=("s_sev", "mean"),
          s_sev_dp=("s_sev", "std"), h2_deteccao_media=("h2_deteccao", "mean"), n=("s_sev", "size")).reset_index()
sev["vies"] = sev.s_sev_media - sev.h2_real
rho = stats.spearmanr(e.s_sev, e.h2_real)
mae = np.mean(np.abs(e.s_sev - e.h2_real))
print(f"Severidade: Spearman rho = {rho.statistic:.3f} (p = {rho.pvalue:.1e}), MAE = {mae:.3f}")
display(sev.round(3))
sev_tipo = e.pivot_table(index="tipo", columns="severidade", values="s_sev", aggfunc="mean", observed=True)[ORD_S]
display(sev_tipo.round(3))

# (b) velocidade vs padrão temporal
e["abrupto_real"] = e.tipo.isin(["abrupto", "recorrente"])
auc_vel = roc_auc_score(e.abrupto_real, e.s_vel)
print(f"Velocidade: AUC (abrupto/recorrente vs gradual/incremental) = {auc_vel:.3f}")
display(e.groupby("tipo", sort=False)[["s_vel", "rise", "atraso_blocos"]].agg(["mean", "std"]).round(3))

# (c) recorrência
tp = ((e.flag_rec) & (e.recorrencia)).sum(); fp_ = ((e.flag_rec) & (~e.recorrencia)).sum()
fn = ((~e.flag_rec) & (e.recorrencia)).sum()
prec = tp/max(tp+fp_, 1); rec = tp/max(tp+fn, 1)
print(f"Recorrência: precisão = {prec:.3f}, revocação = {rec:.3f} (VP={tp}, FP={fp_}, FN={fn})")
display(pd.crosstab([e.severidade, e.recorrencia], e.flag_rec, rownames=["severidade", "recorrência real"], colnames=["sinalizada"]))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for tipo in DRIFT_TYPES:
    g = e[e.tipo == tipo]
    axs[0].scatter(g.h2_real + (DRIFT_TYPES.index(tipo)-1.5)*0.012, g.s_sev, s=10, alpha=0.6, color=COR_TIPO[tipo], label=tipo)
axs[0].plot([0, 0.8], [0, 0.8], color="black", lw=1, ls="--")
axs[0].set_xlabel("$H^2$ real (forma fechada)"); axs[0].set_ylabel("$S_{sev}$ observada (HDDDM)")
axs[0].set_title("Severidade real × observada"); axs[0].legend(fontsize=7)
dados = [e[e.tipo == t].s_vel for t in DRIFT_TYPES]
bp = axs[1].boxplot(dados, patch_artist=True, widths=0.5); axs[1].set_xticks(range(1, 5)); axs[1].set_xticklabels(DRIFT_TYPES)
for patch, t in zip(bp["boxes"], DRIFT_TYPES): patch.set_facecolor(COR_TIPO[t]); patch.set_alpha(0.6)
axs[1].set_ylabel("$S_{vel}$"); axs[1].set_title(f"Velocidade da divergência (AUC = {auc_vel:.2f})")
r_ = e[np.isfinite(e.s_rec)]
for lab, m, c in [("recorrência real", r_.recorrencia, "#e87ba4"), ("estado novo", ~r_.recorrencia, "#8a8a85")]:
    axs[2].hist(r_.s_rec[m], bins=np.linspace(0, 0.8, 41), color=c, alpha=0.7, label=lab)
axs[2].axvline(THETA_REC, color="black", ls="--", lw=1, label=r"$\theta$")
axs[2].set_xlabel("$S_{rec}$ (menor $\\hat H^2$ ao repositório)"); axs[2].set_title("Sinal de recorrência"); axs[2].legend(fontsize=7)
salvar(fig, "fig_sinais_observaveis"); plt.show()

### 8.1 Orientação preliminar de escolha de estratégia

Validação cruzada por semente: metade das repetições (**treino**) define (i) o limiar de velocidade $\tau_v$, (ii) os pontos de corte de severidade observada e (iii) a tabela de recomendação *grupo de tipo × severidade × classificador → estratégia*. Na outra metade (**teste**), cada stream é classificada **apenas pelos sinais** — sem *ground truth* — e a estratégia recomendada é comparada a: o oráculo (melhor estratégia conhecendo tipo e severidade reais), a prática comum (sempre reset) e a melhor estratégia em retrospecto.

In [ ]:
GRUPO = {"abrupto": "abrupto", "gradual": "gradual/incremental", "incremental": "gradual/incremental", "recorrente": "recorrente"}
reps_tr = list(range(N_REPS // 2)); reps_te = list(range(N_REPS // 2, N_REPS))
e_tr = e[e.rep.isin(reps_tr)]

# (i) limiar de velocidade: maximiza a acurácia balanceada no treino
cands = np.linspace(0.3, 1.0, 71)
bal = [np.mean([(e_tr.s_vel[e_tr.abrupto_real] >= c).mean(), (e_tr.s_vel[~e_tr.abrupto_real] < c).mean()]) for c in cands]
TAU_V = float(cands[int(np.argmax(bal))])
# (ii) cortes de severidade: pontos médios entre médias observadas por nível
mu_lvl = e_tr.groupby("severidade", sort=False).s_sev.mean().reindex(ORD_S).values
CORTES = (mu_lvl[:-1] + mu_lvl[1:]) / 2
print(f"tau_v = {TAU_V:.2f}; cortes de severidade observada = {np.round(CORTES, 3)}")

def classificar_stream(ep):
    sig = ep[ep.s_sev >= THETA_REC].sort_values("bloco")
    if sig.empty: return None, None
    grupo = "recorrente" if sig.flag_rec.any() else ("abrupto" if sig.s_vel.iloc[0] >= TAU_V else "gradual/incremental")
    nivel = ORD_S[int(np.searchsorted(CORTES, sig.s_sev.iloc[0]))]
    return grupo, nivel

# (iii) tabela de recomendação por grupo (treino)
rt = res[res.rep.isin(reps_tr)].assign(grupo=lambda d: d.tipo.astype(str).map(GRUPO))
tab_rec = (rt[rt.estrategia != "controle"].groupby(["grupo", "severidade", "classificador", "estrategia"], observed=True)
           .acc_pos.mean().reset_index().sort_values("acc_pos", ascending=False)
           .groupby(["grupo", "severidade", "classificador"], observed=True).head(1))
REC = {(r.grupo, r.severidade, r.classificador): r.estrategia for r in tab_rec.itertuples()}
ORACULO = (rt[rt.estrategia != "controle"].groupby(["tipo", "severidade", "classificador", "estrategia"], observed=True)
           .acc_pos.mean().reset_index().sort_values("acc_pos", ascending=False)
           .groupby(["tipo", "severidade", "classificador"], observed=True).head(1)
           .set_index(["tipo", "severidade", "classificador"]).estrategia.to_dict())
display(tab_rec.pivot_table(index=["grupo", "severidade"], columns="classificador", values="estrategia", aggfunc="first", observed=True))

# avaliação no teste
rte = res[res.rep.isin(reps_te)].set_index(["tipo", "severidade", "classificador", "rep", "estrategia"]).acc_pos
linhas = []
for (tipo, nivel, rep), ep in eps_all[eps_all.rep.isin(reps_te)].groupby(["tipo", "severidade", "rep"]):
    grupo_p, nivel_p = classificar_stream(ep)
    for clf in CLASSIFIERS:
        est_p = REC[(grupo_p, nivel_p, clf)] if grupo_p else "controle"
        get = lambda est: rte.loc[(tipo, nivel, clf, rep, est)]
        todas = {s: get(s) for s in STRATEGIES}
        linhas.append(dict(tipo=tipo, severidade=nivel, classificador=clf, rep=rep, grupo_previsto=grupo_p,
                           nivel_previsto=nivel_p, grupo_ok=grupo_p == GRUPO[tipo], nivel_ok=nivel_p == nivel,
                           est_recomendada=est_p, acc_sinais=todas[est_p], acc_oraculo=todas[ORACULO[(tipo, nivel, clf)]],
                           acc_sempre_reset=todas["reset"], acc_controle=todas["controle"], acc_retrospecto=max(todas.values())))
guia = pd.DataFrame(linhas)
guia.to_csv(f"{OUT}/orientacao_teste.csv", index=False)
resumo_guia = pd.DataFrame({
    "política": ["sinais observáveis (proposta)", "oráculo (tipo e severidade reais)", "sempre reset", "sem retreinamento", "melhor em retrospecto"],
    "acurácia pós-drift": [guia.acc_sinais.mean(), guia.acc_oraculo.mean(), guia.acc_sempre_reset.mean(), guia.acc_controle.mean(), guia.acc_retrospecto.mean()]})
resumo_guia["regret vs retrospecto"] = guia.acc_retrospecto.mean() - resumo_guia["acurácia pós-drift"]
print(f"Acerto do grupo de tipo pelos sinais: {guia.drop_duplicates(['tipo','severidade','rep']).grupo_ok.mean():.2%} | "
      f"acerto do nível de severidade: {guia.drop_duplicates(['tipo','severidade','rep']).nivel_ok.mean():.2%}")
display(resumo_guia.round(4))
w = stats.wilcoxon(guia.acc_sinais, guia.acc_sempre_reset)
print(f"Wilcoxon sinais vs sempre-reset: p = {w.pvalue:.3g}; diferença média = {(guia.acc_sinais-guia.acc_sempre_reset).mean():+.4f}")
display(pd.crosstab(guia.drop_duplicates(['tipo','severidade','rep']).tipo, guia.drop_duplicates(['tipo','severidade','rep']).grupo_previsto.fillna("nenhum")))

## 9. Exportação: tabelas LaTeX e pacote de resultados

In [ ]:
NOME_T = {"abrupto": "Abrupto", "gradual": "Gradual", "incremental": "Incremental", "recorrente": "Recorrente"}
NOME_S = {"baixa": "Baixa", "media": "Média", "alta": "Alta"}
ROT_TAB = {"reset": "Reset", "janela": "Janela", "incremental": "Incremental", "reativacao": "Reativação", "controle": "Controle"}
def dec(x, n=3):
    return "--" if pd.isna(x) else f"{x:.{n}f}".replace(".", ",")
def pval(p):
    return "$<$ 0,001" if p < 0.001 else dec(p, 3)
def to_tex(df, nome, caption, label, colfmt=None, nota=None):
    # grava tabela LaTeX (booktabs, vírgula decimal, cabeçalho único)
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]): df[c] = df[c].map(dec)
    colfmt = colfmt or "l" * (df.shape[1])
    s = df.to_latex(index=False, escape=False, column_format=colfmt)
    corpo = "\\begin{tabular}" + s.split("\\begin{tabular}")[1].rstrip()
    if df.shape[1] >= 6:                       # tabelas largas: reduz só se exceder a largura do texto (pacote adjustbox)
        corpo = "\\begin{adjustbox}{max width=\\textwidth}\n" + corpo + "\n\\end{adjustbox}"
    out = ("\\begin{table}[htbp]\n\\centering\n\\small\n\\caption{" + caption + "}\n\\label{" + label + "}\n"
           + corpo + "\n")
    if nota: out += "\\par\\vspace{2pt}{\\footnotesize " + nota + "}\n"
    out += "\\end{table}\n"
    open(f"{OUT}/tabelas/{nome}.tex", "w").write(out)
def com_grupos(df, col):
    # mostra o valor de `col` só na primeira linha de cada grupo
    df = df.copy(); df[col] = df[col].where(df[col].ne(df[col].shift()), ""); return df

# acurácia pós-drift por classificador
for clf in CLASSIFIERS:
    t = (media[media.classificador == clf].pivot_table(index=["tipo", "severidade"], columns="estrategia", values="acc_pos", observed=True)[ORD_E]
         .reset_index())
    best = t[ORD_E[:-1]].max(axis=1)
    for e in ORD_E:
        t[e] = [("\\textbf{" + dec(v) + "}") if (e != "controle" and np.isclose(v, b_)) else dec(v) for v, b_ in zip(t[e], best)]
    t["tipo"] = t.tipo.astype(str).map(NOME_T); t["severidade"] = t.severidade.astype(str).map(NOME_S)
    t = com_grupos(t, "tipo").rename(columns={"tipo": "Tipo", "severidade": "Severidade", **ROT_TAB})
    to_tex(t, f"tab_acc_{clf}", f"Acurácia prequential pós-drift média por condição -- {clf}", f"tab:acc-{clf.lower()}",
           colfmt="ll" + "c"*5, nota="Média de " + str(N_REPS) + " repetições. Em negrito, a melhor estratégia de cada linha (excluído o controle).")

# ANOVA
TERMO = {"tipo": "Tipo", "severidade": "Severidade", "classificador": "Classificador", "estrategia": "Estratégia"}
an_t = an_show[["df", "F", "PR(>F)", "eta2_parcial"]].reset_index().rename(columns={"index": "termo"})
an_t["termo"] = an_t.termo.map(lambda x: " $\\times$ ".join(TERMO[p] for p in x.split(":")))
an_t = pd.DataFrame({"Termo": an_t.termo, "gl": an_t.df.astype(int).astype(str), "F": an_t.F.map(lambda v: dec(v, 2)),
                     "$p$": an_t["PR(>F)"].map(pval), "$\\eta^2_p$": an_t.eta2_parcial.map(dec)})
to_tex(an_t, "tab_anova", "ANOVA fatorial da acurácia prequential pós-drift", "tab:anova", colfmt="lrrrr",
       nota=f"$R^2$ = {dec(mod.rsquared)}; n = {len(res)} execuções.")

# Wilcoxon par a par
w_t = pd.DataFrame({"Estratégia A": wil.A.map(ROTULO), "Estratégia B": wil.B.map(ROTULO),
                    "Mediana (A $-$ B)": wil.mediana_dif.map(dec), "$p$ (Holm)": wil.p_holm.map(pval)})
to_tex(w_t, "tab_wilcoxon", "Comparações par a par entre estratégias (teste de Wilcoxon pareado, correção de Holm)",
       "tab:wilcoxon", colfmt="llrr", nota=f"Friedman: $\\chi^2$ = {dec(fr.statistic, 1)}, $p$ {pval(fr.pvalue)}; n = {len(wide)} blocos pareados.")

# detector
d_t = tab_det.copy()
d_t["tipo"] = d_t.tipo.map(NOME_T); d_t["severidade"] = d_t.severidade.map(NOME_S)
d_t = com_grupos(d_t, "tipo").rename(columns={"tipo": "Tipo", "severidade": "Severidade", "taxa_deteccao": "Taxa de detecção",
        "atraso_medio_blocos": "Atraso (blocos)", "deteccoes_por_stream": "Detecções/stream", "falsos_alarmes_pre": "Falsos alarmes pré-drift"})
to_tex(d_t, "tab_deteccao", "Desempenho do HDDDM por tipo e severidade do drift", "tab:deteccao", colfmt="llrrrr")

# severidade real x observada
s_t = pd.DataFrame({"Nível": sev.severidade.map(NOME_S), "$H^2$ real": sev.h2_real.map(lambda v: dec(v, 1)),
                    "$S_{sev}$ média": sev.s_sev_media.map(dec), "DP": sev.s_sev_dp.map(dec),
                    "Viés": sev.vies.map(dec), "$\\hat H^2$ na detecção": sev.h2_deteccao_media.map(dec), "n": sev.n.astype(str)})
to_tex(s_t, "tab_severidade_obs", "Severidade real versus severidade observada pelo HDDDM", "tab:severidade-obs", colfmt="lrrrrrr",
       nota=f"Spearman $\\rho$ = {dec(rho.statistic)}; erro absoluto médio = {dec(mae)}.")

# orientação
o_t = pd.DataFrame({"Política": resumo_guia["política"].str.capitalize(),
                    "Acurácia pós-drift": resumo_guia["acurácia pós-drift"].map(dec),
                    "Perda vs. retrospecto": resumo_guia["regret vs retrospecto"].map(dec)})
to_tex(o_t, "tab_orientacao", "Acurácia das políticas de escolha de estratégia nas repetições de teste", "tab:orientacao", colfmt="lrr")

# custo
custo = res.groupby("estrategia", observed=True)[["taxa_acionamento", "n_treinos", "n_reativacoes", "tempo_treino_s", "t_recuperacao", "recuperou"]].mean().loc[ORD_E]
c_t = pd.DataFrame({"Estratégia": [ROTULO[e] for e in custo.index], "Acion./bloco": custo.taxa_acionamento.map(dec),
                    "Treinos": custo.n_treinos.map(lambda v: dec(v, 1)), "Reativações": custo.n_reativacoes.map(lambda v: dec(v, 1)),
                    "Treino (s)": custo.tempo_treino_s.map(dec), "Recuperação (inst.)": custo.t_recuperacao.map(lambda v: f"{v:,.0f}".replace(",", ".")),
                    "Recuperou (\\%)": (100*custo.recuperou).map(lambda v: dec(v, 1))})
to_tex(c_t, "tab_custo", "Custo e recuperação por estratégia (média sobre todas as condições)", "tab:custo", colfmt="lrrrrrr",
       nota="Recuperação: instâncias após o fim da transição até a acurácia do bloco voltar a até 3 p.p. da acurácia pré-drift (censurada no fim do segmento).")

resumo = dict(N_REPS=N_REPS, theta_rec=THETA_REC, tau_v=TAU_V, cortes_sev=CORTES.tolist(),
              anova_R2=mod.rsquared, friedman_chi2=fr.statistic, friedman_p=fr.pvalue, ranks=ranks.to_dict(),
              spearman_sev=rho.statistic, mae_sev=mae, auc_vel=auc_vel, rec_precisao=prec, rec_revocacao=rec,
              politicas=resumo_guia.set_index("política")["acurácia pós-drift"].to_dict())
json.dump(resumo, open(f"{OUT}/resumo.json", "w"), indent=2, ensure_ascii=False, default=float)
print(json.dumps(resumo, indent=2, ensure_ascii=False, default=float))

## 10. Análise de sensibilidade (cadência da janela e limiar do detector)

As conclusões dependem de dois parâmetros de projeto: a cadência da janela deslizante (`K_SLIDE`) e o parâmetro $\gamma$ do limiar do HDDDM. Esta seção repete o fatorial com 5 repetições variando cada um isoladamente.

In [ ]:
RODAR_SENSIBILIDADE = True
REPS_SENS = 5
def executar_sens(tipo, nivel, rep, k_slide, gamma):
    ti, si = DRIFT_TYPES.index(tipo), list(SEVERITY).index(nivel)
    seed = 1000*rep + 10*ti + si
    X, y, mu, trans = gerar_stream(tipo, SEVERITY[nivel], seed)
    eventos, _, _ = rodar_detector(X, gamma=gamma)
    out = []
    for clf in CLASSIFIERS:
        for est in STRATEGIES:
            acc, nt, nr, _ = rodar_estrategia(X, y, eventos, clf, est, seed, THETA_REC, k_slide=k_slide)
            out.append(dict(tipo=tipo, severidade=nivel, rep=rep, classificador=clf, estrategia=est,
                            k_slide=k_slide, gamma=gamma, acc_pos=metricas(acc, trans, tipo)["acc_pos"],
                            taxa_acionamento=(nt+nr)/(N_BLOCKS-1)))
    return out
if RODAR_SENSIBILIDADE:
    grade = [(K_SLIDE, g) for g in (1.5, 2.0, 2.5)] + [(k, HDDDM_GAMMA) for k in (2, 8, 16)]
    tarefas_s = [(t, s, r, k, g) for (k, g) in grade for t in DRIFT_TYPES for s in SEVERITY for r in range(REPS_SENS)]
    sens = pd.DataFrame([l for o in Parallel(n_jobs=N_JOBS)(delayed(executar_sens)(*a) for a in tarefas_s) for l in o])
    sens.to_csv(f"{OUT}/sensibilidade.csv", index=False)
    tab_sens = sens.pivot_table(index=["k_slide", "gamma"], columns="estrategia", values="acc_pos")[ORD_E]
    tab_sens["acion_janela"] = sens[sens.estrategia == "janela"].groupby(["k_slide", "gamma"]).taxa_acionamento.mean()
    display(tab_sens.round(3))
    ts = tab_sens.reset_index()
    ts = pd.DataFrame({"Cadência $K$": ts.k_slide.astype(str), "$\\gamma$": ts.gamma.map(lambda v: dec(v, 1)),
                       **{ROT_TAB[e]: ts[e].map(dec) for e in ORD_E}, "Acion. janela": ts.acion_janela.map(dec)})
    to_tex(ts, "tab_sensibilidade", "Sensibilidade da acurácia pós-drift média à cadência da janela deslizante e ao limiar do HDDDM",
           "tab:sensibilidade", colfmt="cc" + "r"*6, nota=f"{REPS_SENS} repetições por condição; configuração principal: $K$ = {K_SLIDE}, $\\gamma$ = {dec(HDDDM_GAMMA, 1)}.")

## 11. Pacote de resultados

In [ ]:
import shutil; shutil.make_archive("resultados", "zip", OUT)
print("arquivos:", sorted(os.listdir(f"{OUT}/tabelas")) + sorted(os.listdir(f"{OUT}/figuras")))
try:
    from google.colab import files; files.download("resultados.zip")
except Exception:
    pass